In [0]:
# Cargar la tabla que encontraste en el Catalog Explorer
df = spark.read.table("samples.nyctaxi.trips")

# Ver los datos
display(df)

In [0]:
from pyspark.sql.functions import rand, col
from pyspark.sql import functions as F

# Crear 1 millón de filas
df = spark.range(0, 1000000)

# Agregar columnas
df = df.withColumn("categoria", (rand()*10).cast("int")) \
       .withColumn("valor", (rand()*100).cast("int"))

df.show(5)

In [0]:
##genera shuffle

df_grouped = df.groupBy("categoria").agg(F.sum("valor").alias("total"))

df_grouped.explain(True)

In [0]:
df1 = spark.range(0, 1000000).withColumn("key", (rand()*100).cast("int"))
df2 = spark.range(0, 100).withColumn("key", col("id"))

df2 = df2.drop("id")

In [0]:
joined = df1.join(df2, "key")

joined.explain(True)

In [0]:
from pyspark.sql.functions import broadcast

joined_broadcast = df1.join(broadcast(df2), "key")

joined_broadcast.explain(True)

In [0]:
##df.rdd.getNumPartitions()


df.explain()

In [0]:
df_rep = df.repartition(10)
df_rep.explain()

In [0]:
df_coal = df.coalesce(2)
df_coal.explain()

In [0]:
from pyspark.sql.functions import lit

# Crear skew: muchas filas con misma clave
df_skew = spark.range(0, 1000000) \
    .withColumn("categoria",
                F.when(col("id") < 900000, lit(1))
                 .otherwise((rand()*10).cast("int")))

df_skew_group = df_skew.groupBy("categoria").count()

df_skew_group.explain(True)
